## Bloco 0 - Autenticação
> Colab pedindo permissão pra agir com a sua conta Google. Pro



In [ ]:
# %% 0. Autenticação com sua conta Google (roda só dentro do Colab)
from google.colab import auth
auth.authenticate_user()
print("Autenticado com sucesso.")

# %% 1. Escolha a fonte de dados
# "sheets"    -> lê de uma planilha Google Sheets (mais simples de começar)
# "bigquery"  -> lê direto do BigQuery, se o fechamento já estiver lá
# "sintetico" -> gera dado fake, só para testar o pipeline sem acesso ainda
FONTE_DADOS = "sheets"

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.cluster import KMeans

pd.set_option("display.width", 120)
np.random.seed(42)

Autenticado com sucesso.


## Bloco 01 - Simulação de 3 ano de fechamento em G&A



In [ ]:
if FONTE_DADOS == "sintetico":
    import gspread
    from google.auth import default

    creds, _ = default()
    gc = gspread.authorize(creds)

    # Troque pela URL real da sua planilha de fechamento
    PLANILHA_URL = "https://docs.google.com/spreadsheets/d/SEU_ID_AQUI/edit"
    ABA = "Fechamento"  # nome da aba com as colunas: departamento, conta, ano_mes, orcado, real

    sh = gc.open_by_url(PLANILHA_URL)
    ws = sh.worksheet(ABA)
    df = pd.DataFrame(ws.get_all_records())
    df["orcado"] = df["orcado"].astype(float)
    df["real"] = df["real"].astype(float)

# ---------- Opção B: BigQuery ----------
elif FONTE_DADOS == "sintetico":
    from google.cloud import bigquery

    PROJECT_ID = "seu-projeto-gcp"
    query = """
        SELECT departamento, conta, ano_mes, orcado, real
        FROM `seu-projeto-gcp.seu_dataset.fechamento_ga`
    """
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query).to_dataframe()

# ---------- Opção C: sintético (só para testar o pipeline agora) ----------
else:
    departamentos = ["Comercial", "Operações", "TI", "Marketing", "Financeiro", "RH"]
    contas = ["Passagens aéreas", "Hospedagem", "Licenças de software", "Cloud computing",
              "Consultoria externa", "Treinamentos", "Material gráfico", "Transporte local",
              "Eventos e feiras", "Alimentação"]
    meses = pd.period_range("2023-01", "2025-12", freq="M")
    linhas = []
    for depto in departamentos:
        for conta in np.random.choice(contas, 4, replace=False):
            base_orc = np.random.randint(8000, 60000)
            estrutural = np.random.choice([0, 1], p=[0.6, 0.4])
            for mes in meses:
                sazonal = 1.5 if mes.month in [11, 12] and conta in ["Eventos e feiras", "Passagens aéreas"] else 1.0
                ruido = np.random.normal(1.0, 0.12)
                real = base_orc * sazonal * (1.3 if estrutural else 1.0) * ruido
                linhas.append([depto, conta, str(mes), round(base_orc, 2), round(real, 2)])
    df = pd.DataFrame(linhas, columns=["departamento", "conta", "ano_mes", "orcado", "real"])

print(df.shape)
print(df.head())

(864, 5)
  departamento             conta  ano_mes  orcado      real
0    Comercial  Eventos e feiras  2023-01   24850  18853.31
1    Comercial  Eventos e feiras  2023-02   24850  23380.46
2    Comercial  Eventos e feiras  2023-03   24850  26020.67
3    Comercial  Eventos e feiras  2023-04   24850  22079.17
4    Comercial  Eventos e feiras  2023-05   24850  25088.06


In [ ]:
df[(df["conta"]=="Eventos e feiras") & (df["ano_mes"].isin(["2023-11","2023-12"]))]

,departamento,conta,ano_mes,orcado,real
10,Comercial,Eventos e feiras,2023-11,24850,38268.36
11,Comercial,Eventos e feiras,2023-12,24850,33839.84
334,TI,Eventos e feiras,2023-11,36295,62746.97
335,TI,Eventos e feiras,2023-12,36295,59478.63
550,Marketing,Eventos e feiras,2023-11,44455,74444.97
551,Marketing,Eventos e feiras,2023-12,44455,72450.05
622,Financeiro,Eventos e feiras,2023-11,32917,57602.93
623,Financeiro,Eventos e feiras,2023-12,32917,33618.68


## Bloco 02 - Transformando tabelas e criando colunas

In [ ]:
df = df.sort_values(["departamento", "conta", "ano_mes"])
df["desvio_alto"] = (df["delta_pct"].abs() > 15).astype(int)
df["recorrencia_6m"] = (
    df.groupby(["departamento", "conta"])["desvio_alto"]
    .transform(lambda s: s.rolling(6, min_periods=1).sum())
)
df["volatilidade_conta"] = (
    df.groupby(["departamento", "conta"])["delta_pct"]
    .transform(lambda s: s.expanding(min_periods=3).std())
    .fillna(0)
)
df["mes_num"] = df["ano_mes"].astype(str).str.slice(5, 7).astype(int)

df[df["conta"]=="Eventos e feiras"].head(15)


,departamento,conta,ano_mes,orcado,real,delta_pct,desvio_alto,recorrencia_6m,volatilidade_conta,mes_num
0,Comercial,Eventos e feiras,2023-01,24850,18853.31,-24.131549,1,1.0,0.000000,1
1,Comercial,Eventos e feiras,2023-02,24850,23380.46,-5.913642,0,1.0,0.000000,2
2,Comercial,Eventos e feiras,2023-03,24850,26020.67,4.710946,0,1.0,14.586886,3
3,Comercial,Eventos e feiras,2023-04,24850,22079.17,-11.150221,0,1.0,11.986718,4
4,Comercial,Eventos e feiras,2023-05,24850,25088.06,0.957988,0,1.0,11.317187,5
5,Comercial,Eventos e feiras,2023-06,24850,24374.32,-1.914205,0,1.0,10.341867,6
6,Comercial,Eventos e feiras,2023-07,24850,24916.27,0.266680,0,0.0,9.755862,7
7,Comercial,Eventos e feiras,2023-08,24850,23574.32,-5.133521,0,0.0,9.032390,8
8,Comercial,Eventos e feiras,2023-09,24850,23264.12,-6.381811,0,0.0,8.456884,9
9,Comercial,Eventos e feiras,2023-10,24850,24499.69,-1.409698,0,0.0,8.072951,10


## Bloco 03 - A regra de "desvio relevante"

In [ ]:
limite = df["delta_pct"].abs().quantile(0.80)
df["relevante"] = (df["delta_pct"].abs() >= limite).astype(int)
print(f"\nLimite (percentil 80): {limite:.1f}% | Taxa de positivos: {df['relevante'].mean():.2%}")


Limite (percentil 80): 35.6% | Taxa de positivos: 20.02%


## Bloco 04 - O Classificador

In [ ]:
features = ["orcado", "recorrencia_6m", "volatilidade_conta", "mes_num"] ## features são as "pistas" que o modelo vai usar pra tentar adivinhar se é relevante.
X = pd.get_dummies(df[features + ["departamento"]], columns=["departamento"], drop_first=True)
y = df["relevante"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

baseline = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print("Baseline (logística) AUC:", roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1]))

modelo = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, class_weight="balanced")
modelo.fit(X_train, y_train)
print("Random Forest AUC:", roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1]))
print(classification_report(y_test, modelo.predict(X_test)))

df["score"] = (modelo.predict_proba(X)[:, 1] * 100).round(0).astype(int)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Baseline (logística) AUC: 0.8171797284581261
Random Forest AUC: 0.8460814625621723
              precision    recall  f1-score   support

           0       0.96      0.78      0.86       173
           1       0.50      0.88      0.64        43

    accuracy                           0.80       216
   macro avg       0.73      0.83      0.75       216
weighted avg       0.87      0.80      0.82       216



## Bloco 05 - Clustering

In [ ]:
perfil_conta = df.groupby(["departamento", "conta"]).agg(
    freq_desvio=("desvio_alto", "mean"),
    media_delta=("delta_pct", "mean"),
    volatilidade=("delta_pct", "std"),
).fillna(0)

X_cluster = StandardScaler().fit_transform(perfil_conta)
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
perfil_conta["cluster"] = kmeans.fit_predict(X_cluster)

ordem = perfil_conta.groupby("cluster")["freq_desvio"].mean().sort_values(ascending=False)
mapa_label = {ordem.index[0]: "Estrutural", ordem.index[1]: "Sazonal", ordem.index[2]: "Pontual"}
perfil_conta["perfil"] = perfil_conta["cluster"].map(mapa_label)

perfil_conta[["freq_desvio", "media_delta", "perfil"]]
perfil_reset = perfil_conta.reset_index()[["departamento", "conta", "perfil"]]
df = df.merge(perfil_reset, on=["departamento", "conta"], how="left")

In [ ]:
tabela_final = (
    df[df["ano_mes"] == df["ano_mes"].max()]
    [["departamento", "conta", "real", "orcado", "delta_pct", "score", "perfil"]]
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)
print("\nTabela final (último mês, ordenada por score):")
print(tabela_final.head(10))


Tabela final (último mês, ordenada por score):
  departamento             conta      real  orcado  delta_pct  score      perfil
0    Marketing  Eventos e feiras  68384.15   44455  53.827803     87     Sazonal
1    Comercial  Passagens aéreas  15418.58    9636  60.010170     81     Sazonal
2   Financeiro  Material gráfico  63492.08   42911  47.962247     79  Estrutural
3   Financeiro  Eventos e feiras  48017.38   32917  45.874108     79     Sazonal
4           RH   Cloud computing  80423.49   54600  47.295769     79  Estrutural
5           TI  Material gráfico  70937.02   51125  38.752117     77  Estrutural
6    Comercial        Hospedagem  50893.73   40606  25.335492     76  Estrutural
7    Comercial  Eventos e feiras  36507.30   24850  46.910664     72     Sazonal
8    Marketing       Alimentação  16862.15   16258   3.716017     70  Estrutural
9           RH  Transporte local  30195.98   21737  38.915122     69  Estrutural
